In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/MachineLearningRating_v3.txt', sep='|')
# Create target
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

# Convert to numeric
df['TotalClaims'] = pd.to_numeric(df['TotalClaims'], errors='coerce')
df['TotalPremium'] = pd.to_numeric(df['TotalPremium'], errors='coerce')

# Select columns
selected_columns = [
    'TotalClaims', 'TotalPremium', 'HasClaim',
    'Gender', 'Province', 'MaritalStatus', 'Title',
    'VehicleType', 'make', 'Model', 'RegistrationYear', 
    'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors',
    'CoverType', 'Product', 'SumInsured', 'ExcessSelected',
    'AlarmImmobiliser', 'TrackingDevice'
]

existing_columns = [col for col in selected_columns if col in df.columns]
df_clean = df[existing_columns].copy()

print("Identify numeric vs categorical columns")

# Define which columns are truly numeric
numeric_cols_true = ['RegistrationYear', 'Cylinders', 'cubiccapacity', 
                     'kilowatts', 'NumberOfDoors', 'SumInsured', 'TotalClaims', 'TotalPremium']

# Define which columns are categorical (even if they contain numbers)
categorical_cols_true = ['Gender', 'Province', 'MaritalStatus', 'Title', 'VehicleType', 
                         'make', 'Model', 'bodytype', 'CoverType', 'Product', 
                         'ExcessSelected', 'AlarmImmobiliser', 'TrackingDevice']

print(f"Numeric columns: {numeric_cols_true}")
print(len(numeric_cols_true))
print(f"Categorical columns: {categorical_cols_true}")
print(len(categorical_cols_true))

Identify numeric vs categorical columns
Numeric columns: ['RegistrationYear', 'Cylinders', 'cubiccapacity', 'kilowatts', 'NumberOfDoors', 'SumInsured', 'TotalClaims', 'TotalPremium']
8
Categorical columns: ['Gender', 'Province', 'MaritalStatus', 'Title', 'VehicleType', 'make', 'Model', 'bodytype', 'CoverType', 'Product', 'ExcessSelected', 'AlarmImmobiliser', 'TrackingDevice']
13


In [18]:
df.columns

Index(['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth',
       'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language',
       'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province',
       'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode',
       'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders',
       'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors',
       'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser',
       'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff',
       'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet',
       'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm',
       'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section',
       'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium',
       'TotalClaims', 'HasClaim'],
      dtype='str')

In [28]:
#Fill missing values - NUMERIC columns"

for col in numeric_cols_true:
    if col in df_clean.columns:
        missing_count = df_clean[col].isnull().sum()
        if missing_count > 0:
            median_val = df_clean[col].median()
            df_clean.loc[df_clean[col].isnull(), col] = median_val
            print(f"Filled {col}: {missing_count:,} missing values with median: {median_val}")
        else:
            print(f"{col}: no missing values")

RegistrationYear: no missing values
Cylinders: no missing values
cubiccapacity: no missing values
kilowatts: no missing values
NumberOfDoors: no missing values
SumInsured: no missing values
TotalClaims: no missing values
TotalPremium: no missing values


In [29]:
#Fill remaining CATEGORICAL columns
# List of categorical columns that still have missing values
remaining_categorical = ['Gender', 'MaritalStatus', 'VehicleType', 'make', 'Model', 'bodytype']

for col in remaining_categorical:
    if col in df_clean.columns:
        # Check current missing count
        missing_before = df_clean[col].isnull().sum()
        print(f"\n{col}: {missing_before:,} missing values before")
        
        # Convert to string first
        df_clean[col] = df_clean[col].astype(str)
        
        # Replace 'nan' string with 'Unknown'
        df_clean.loc[df_clean[col] == 'nan', col] = 'Unknown'
        
        # Also fill any actual NaN values
        df_clean.loc[df_clean[col].isnull(), col] = 'Unknown'
        
        # Verify
        missing_after = (df_clean[col] == 'Unknown').sum()
        print(f"  ✓ Filled: {missing_after:,} values set to 'Unknown'")


Gender: 9,536 missing values before
  ✓ Filled: 9,536 values set to 'Unknown'

MaritalStatus: 8,259 missing values before
  ✓ Filled: 8,259 values set to 'Unknown'

VehicleType: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'

make: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'

Model: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'

bodytype: 552 missing values before
  ✓ Filled: 552 values set to 'Unknown'


In [30]:
#Final Verification"
# Check all columns for missing values
missing_check = df_clean.isnull().sum()
missing_cols = missing_check[missing_check > 0]

if len(missing_cols) == 0:
    print("No missing values in any column!")
else:
    print(f"Still have missing values in: {missing_cols.index.tolist()}")
    print(missing_cols)

# Also check for 'nan' strings
print("Checking for 'nan' strings:")
for col in df_clean.select_dtypes(include=['object']).columns:
    nan_count = (df_clean[col] == 'nan').sum()
    if nan_count > 0:
        print(f"{col}: {nan_count} 'nan' strings found - fixing...")
        df_clean.loc[df_clean[col] == 'nan', col] = 'Unknown'
    else:
        print(f"{col}: clean")

print(f"\nFinal shape: {df_clean.shape}")
print(f"Total missing values: {df_clean.isnull().sum().sum()}")

No missing values in any column!
Checking for 'nan' strings:
Gender: clean
Province: clean
MaritalStatus: clean
Title: clean
VehicleType: clean
make: clean
Model: clean
bodytype: clean
CoverType: clean
Product: clean
ExcessSelected: clean
AlarmImmobiliser: clean
TrackingDevice: clean

Final shape: (1000098, 22)
Total missing values: 0


# Data Preparation & Cleaning Summary

## Overview

The raw dataset `MachineLearningRating_v3.txt` contained **1,000,098 rows** and **52 columns**. Significant data cleaning was required to prepare the data for predictive modeling. This document outlines the cleaning steps performed.

---

## Initial Data Assessment

| Metric | Value |
|--------|-------|
| Total rows | 1,000,098 |
| Total columns | 52 |
| File format | Pipe-delimited (`|`) |
| Time period | Feb 2014 – Aug 2015 |

### Column Types Identified

| Type | Count | Examples |
|------|-------|----------|
| Numeric | 15 | TotalPremium, TotalClaims, RegistrationYear, Cylinders |
| Categorical | 35 | Gender, Province, VehicleType, make, Model, CoverType |
| Boolean | 1 | IsVATRegistered |
| Date | 1 | TransactionMonth |

---

## Missing Value Analysis

### Columns with >50% Missing Values (Dropped)

These columns were **removed** because they lacked sufficient data for meaningful analysis:

| Column | Missing % | Reason for Dropping |
|--------|-----------|---------------------|
| CrossBorder | 99.93% | Almost all values missing |
| NumberOfVehiclesInFleet | 100% | Completely empty |
| WrittenOff | 64.18% | Majority missing |
| Rebuilt | 64.18% | Majority missing |
| Converted | 64.18% | Majority missing |
| CustomValueEstimate | 77.96% | Majority missing |

### Columns with <50% Missing Values (Kept & Filled)

These columns were **retained** and missing values were imputed:

| Column | Missing Count | Missing % | Handling Method |
|--------|---------------|-----------|-----------------|
| Bank | 145,961 | 14.59% | Dropped (not needed for modeling) |
| AccountType | 40,232 | 4.02% | Dropped (not needed for modeling) |
| Gender | 9,536 | 0.95% | Filled with 'Unknown' |
| MaritalStatus | 8,259 | 0.83% | Filled with 'Unknown' |
| VehicleType | 552 | 0.06% | Filled with 'Unknown' |
| make | 552 | 0.06% | Filled with 'Unknown' |
| Model | 552 | 0.06% | Filled with 'Unknown' |
| bodytype | 552 | 0.06% | Filled with 'Unknown' |
| Cylinders | 552 | 0.06% | Filled with median (4.0) |
| cubiccapacity | 552 | 0.06% | Filled with median (2694.0) |
| kilowatts | 552 | 0.06% | Filled with median (111.0) |
| NumberOfDoors | 552 | 0.06% | Filled with median (4.0) |
| NewVehicle | 153,295 | 15.33% | Dropped (not needed for modeling) |

---

## Columns Selected for Modeling

After cleaning, **22 columns** were retained for modeling:

### Target Variables

| Column | Description | Type |
|--------|-------------|------|
| `TotalClaims` | Claim amount (Rands) | Numeric |
| `TotalPremium` | Premium paid (Rands) | Numeric |
| `HasClaim` | Binary indicator (1 = claim occurred) | Binary |

### Feature Variables

| Category | Columns |
|----------|---------|
| **Demographics** | Gender, Province, MaritalStatus, Title |
| **Vehicle Information** | VehicleType, make, Model, RegistrationYear, Cylinders, cubiccapacity, kilowatts, bodytype, NumberOfDoors |
| **Policy Information** | CoverType, Product, SumInsured, ExcessSelected |
| **Safety Features** | AlarmImmobiliser, TrackingDevice |

---

## Missing Value Imputation Strategy

### Numeric Columns

```python
# Filled with median to avoid skewing distributions
df_clean['Cylinders'].fillna(df_clean['Cylinders'].median(), inplace=True)
df_clean['cubiccapacity'].fillna(df_clean['cubiccapacity'].median(), inplace=True)
df_clean['kilowatts'].fillna(df_clean['kilowatts'].median(), inplace=True)
df_clean['NumberOfDoors'].fillna(df_clean['NumberOfDoors'].median(), inplace=True)
```